# Bước 2 — Chia tài liệu thành chunk
Đầu vào: `documents.json`. Đầu ra: `chunks.json`. Notebook cho thấy tác dụng của `chunk_size` và `overlap`.

In [ ]:
from pathlib import Path
import json, re

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
STORAGE_DIR = PROJECT_DIR / 'storage'
documents = json.loads((STORAGE_DIR / 'documents.json').read_text(encoding='utf-8'))
CHUNK_SIZE = 700
CHUNK_OVERLAP = 120
print('Số phần tài liệu:', len(documents))

In [ ]:
def clean_text(text):
    text = text.replace('\x00', ' ')
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def split_text(text, chunk_size=700, overlap=120):
    if overlap >= chunk_size:
        raise ValueError('overlap phải nhỏ hơn chunk_size')
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            natural_end = max(text.rfind('\n', start, end), text.rfind('. ', start, end))
            if natural_end > start + chunk_size // 2:
                end = natural_end + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = end - overlap
    return chunks

## Thử trên một tài liệu
Hai chunk liên tiếp có một phần nội dung lặp lại. Phần lặp giúp thông tin ở ranh giới không bị mất.

In [ ]:
sample = clean_text(documents[0]['content'])
sample_chunks = split_text(sample, CHUNK_SIZE, CHUNK_OVERLAP)
print('Độ dài văn bản:', len(sample))
print('Số chunk:', len(sample_chunks))
print('Chunk 1:', sample_chunks[0])
print('\nChunk 2:', sample_chunks[1] if len(sample_chunks) > 1 else 'Không có')

In [ ]:
chunks = []
for document in documents:
    for part, content in enumerate(split_text(clean_text(document['content']), CHUNK_SIZE, CHUNK_OVERLAP), 1):
        chunks.append({
            'chunk_id': len(chunks), 'content': content,
            'source': document['source'], 'file_type': document['file_type'],
            'location': document['location'], 'part': part
        })

print('Tổng chunk:', len(chunks))
print('Độ dài trung bình:', round(sum(len(x['content']) for x in chunks) / len(chunks)))

In [ ]:
chunks[:2]

In [ ]:
output = STORAGE_DIR / 'chunks.json'
output.write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã lưu:', output)